# 09 · Controlling training: over-fitting, early stopping, regularisation, optimizers

So far "train for N epochs" was good enough. The homework's last task demanded more: **early stopping with automatic restoration of the best weights**. This notebook covers that and the other tools that decide *how* a network is trained: L2/L1 penalties, dropout, and what `SGD`, momentum and `Adam` actually do.

To *see* over-fitting clearly we use a tiny 1-D problem: 40 noisy training points from a wavy curve, and a network that is far too big for them.

---
## Theory in 60 seconds

- **Over-fitting**: the network starts to fit the *noise* of the training points. Training loss keeps falling; validation loss stops falling and rises.
- **Early stopping**: monitor the validation loss; remember the weights of the best epoch; stop when there was no improvement for `patience` epochs; **restore** the best weights.
- **L2 regularisation / weight decay**: add `λ·Σw²` to the loss → large weights are penalised → smoother functions. **L1** (`λ·Σ|w|`) pushes weights to exactly zero.
- **Dropout**: during training each hidden activation is zeroed with probability `p` (and the survivors are scaled up by `1/(1−p)`); at evaluation time dropout does nothing. This is the reason `model.train()` / `model.eval()` exist.
- **Momentum**: the step follows a running sum of past gradients — speeds up along consistent directions, damps zig-zag. **Adam**: momentum + a per-parameter step size that normalises by the recent gradient magnitude.

In [ ]:
import sys; sys.path.insert(0, "..")
import copy
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from dnd_utils import check

NOISE_STD = 0.35
def true_fn(x):
    return np.sin(2 * x) + 0.5 * x

def make_data(n, rng):
    x = rng.uniform(-3, 3, n)
    y = true_fn(x) + rng.normal(0, NOISE_STD, n)
    return torch.tensor(x, dtype=torch.float32)[:, None], torch.tensor(y, dtype=torch.float32)[:, None]

_rng = np.random.default_rng(5)
X_train, y_train = make_data(40, _rng)        # (40, 1), (40, 1)   <- very little data
X_val, y_val = make_data(300, _rng)
X_test, y_test = make_data(300, _rng)

class BigMLP(nn.Module):
    """1 -> 128 -> 128 -> 1, optionally with dropout after each hidden activation."""
    def __init__(self, p_drop=0.0):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(1, 128), nn.ReLU(), nn.Dropout(p_drop),
                                    nn.Linear(128, 128), nn.ReLU(), nn.Dropout(p_drop),
                                    nn.Linear(128, 1))
    def forward(self, x):
        return self.layers(x)

loss_fn = nn.MSELoss()

def full_loss(model, X, y):
    """Loss on a whole split, in evaluation mode, without gradients."""
    model.eval()
    with torch.no_grad():
        return loss_fn(model(X), y).item()

def plot_fit(models: dict):
    grid = torch.linspace(-3, 3, 400)[:, None]
    plt.figure(figsize=(7, 3.5))
    plt.scatter(X_train[:, 0], y_train[:, 0], s=18, color="black", label="training points", zorder=3)
    plt.plot(grid[:, 0], true_fn(grid[:, 0].numpy()), "--", color="gray", label="true function")
    for name, m in models.items():
        m.eval()
        with torch.no_grad():
            plt.plot(grid[:, 0], m(grid)[:, 0], label=name)
    plt.ylim(-3.5, 3.5); plt.legend(); plt.show()

print("parameters:", sum(p.numel() for p in BigMLP().parameters()), "| training points:", len(X_train), "| noise variance:", NOISE_STD ** 2)

Roughly 17 000 parameters for 40 data points. The noise variance `0.1225` is the best MSE any model can achieve on *new* data.

---
## Exercise 9.1 · Watch a network over-fit

Write `fit(model, optimizer, n_epochs, seed=0)`:

- each epoch visits the 40 training points in a new random order, in mini-batches of 10 (use `torch.randperm(40, generator=g)` with the generator given below, and index the tensors — no `DataLoader` needed);
- after each epoch append `full_loss(...)` on the training set and on the validation set to two lists;
- returns `(train_hist, val_hist)`.

Then train a `BigMLP()` with `Adam(lr=0.003)` for 1500 epochs.

In [ ]:
def fit(model, optimizer, n_epochs, seed=0):
    g = torch.Generator().manual_seed(seed)
    train_hist, val_hist = [], []
    raise NotImplementedError
    return train_hist, val_hist

torch.manual_seed(0)
model_plain = BigMLP()
train_hist, val_hist = None, None

In [ ]:
check("1500 epochs recorded", lambda: len(train_hist) == len(val_hist) == 1500)
check("training loss ends BELOW the noise variance 0.1225", lambda: train_hist[-1] < 0.1)
check("validation loss ends clearly above its best value", lambda: val_hist[-1] > 1.15 * min(val_hist))

From the two histories compute `best_epoch` (index of the lowest validation loss), `val_at_best` and `val_at_end`. Then plot both curves (log y-axis, a vertical line at `best_epoch`, legend) and call `plot_fit({"after 1500 epochs": model_plain})`.

In [ ]:
best_epoch = None
val_at_best = None
val_at_end = None
# plots

In [ ]:
check("best_epoch", lambda: best_epoch == int(np.argmin(val_hist)) and val_at_best == min(val_hist) and val_at_end == val_hist[-1])

The training loss ends **below the noise variance**. A model that knew the true function exactly would have a training loss of about 0.12. What does a training loss of 0.04 tell you, and what do you see in the picture of the fitted curve?

✍️ **Your answer** (double-click, write it in your own words *before* opening the solution):

...

---
## Exercise 9.2 · A trap before early stopping: `state_dict()` is not a copy

Early stopping must remember the weights of the best epoch. `model.state_dict()` returns a dictionary of the model's tensors. Predict the two printed values.

In [ ]:
torch.manual_seed(0)
m = nn.Linear(1, 1)
opt = torch.optim.SGD(m.parameters(), lr=0.1)

saved_reference = m.state_dict()                       # "save" attempt 1
saved_copy = copy.deepcopy(m.state_dict())             # "save" attempt 2
weight_at_save_time = m.weight.item()

loss_fn(m(X_train), y_train).backward(); opt.step()    # one training step changes the model

ref_still_has_old_weight = saved_reference["weight"].item() == weight_at_save_time
copy_still_has_old_weight = saved_copy["weight"].item() == weight_at_save_time

In [ ]:
pred_ref_still_old = None         # True / False
pred_copy_still_old = None        # True / False

In [ ]:
check("reference", lambda: pred_ref_still_old is ref_still_has_old_weight)
check("deep copy", lambda: pred_copy_still_old is copy_still_has_old_weight)

So `best_state = model.state_dict()` "saves" nothing: at the end `best_state` simply contains the **final** weights, and `load_state_dict(best_state)` restores the model to… itself. No error, no warning. Always `copy.deepcopy(model.state_dict())`.

---
## Exercise 9.3 · Early stopping

Write `fit_early_stopping(model, optimizer, max_epochs, patience, seed=0)`. Same training as `fit`, plus:

- after each epoch: if the validation loss is **lower than the best so far**, remember it, the epoch index and a snapshot of the weights, and reset the counter; otherwise increase the counter and **stop** when it reaches `patience`;
- after the loop: restore the best weights into `model`;
- returns `(train_hist, val_hist, best_epoch)`.

In [ ]:
def fit_early_stopping(model, optimizer, max_epochs, patience, seed=0):
    raise NotImplementedError

torch.manual_seed(0)
model_es = BigMLP()
es_train_hist, es_val_hist, es_best_epoch = None, None, None      # run with Adam(lr=0.003), max_epochs=1500, patience=100

In [ ]:
check("it stopped early", lambda: len(es_val_hist) < 1500)
check("best_epoch is the argmin of the validation history", lambda: es_best_epoch == int(np.argmin(es_val_hist)))
check("it stopped exactly `patience` epochs after the best one", lambda: len(es_val_hist) == es_best_epoch + 100 + 1)
check("the model holds the BEST weights, not the last ones", lambda: abs(full_loss(model_es, X_val, y_val) - min(es_val_hist)) < 1e-6)
check("same trajectory as the plain run up to the stop (same seeds)", lambda: np.allclose(es_val_hist, val_hist[:len(es_val_hist)], atol=1e-4))

Three questions:

1. Why `patience` at all — why not stop the first time the validation loss goes up?
2. Could we monitor the **test** loss instead of the validation loss?
3. After early stopping, is the *validation* loss of the final model an unbiased estimate of its performance on new data?

✍️ **Your answer** (double-click, write it in your own words *before* opening the solution):

...

---
## Exercise 9.4 · L2 and L1 penalties

**a)** Write `l2_sum(model)` (sum of the squares of **all** parameters) and `l1_sum(model)` (sum of absolute values). Both must return a scalar tensor that is part of the graph.

In [ ]:
def l2_sum(model):
    raise NotImplementedError

def l1_sum(model):
    raise NotImplementedError

In [ ]:
_lin = nn.Linear(2, 1)
with torch.no_grad():
    _lin.weight.copy_(torch.tensor([[3.0, -4.0]])); _lin.bias.copy_(torch.tensor([1.0]))
check("l2_sum = 9 + 16 + 1", lambda: abs(l2_sum(_lin).item() - 26.0) < 1e-6)
check("l1_sum = 3 + 4 + 1", lambda: abs(l1_sum(_lin).item() - 8.0) < 1e-6)
check("both are differentiable", lambda: l2_sum(_lin).requires_grad and l1_sum(_lin).requires_grad)

**b)** `torch.optim.SGD(..., weight_decay=λ)` is documented as adding `λ·w` to every gradient. Adding which penalty term to the loss gives exactly the same update? Decide the factor on paper (differentiate your penalty), then make the check pass: do **one** SGD step (`lr = 0.1`) on `model_b` using an explicit penalty, so that it ends up identical to `model_a`, which uses `weight_decay=0.02`.

In [ ]:
torch.manual_seed(0)
model_a = nn.Linear(1, 1)
model_b = copy.deepcopy(model_a)
lam = 0.02

# model_a: built-in weight decay
opt_a = torch.optim.SGD(model_a.parameters(), lr=0.1, weight_decay=lam)
loss_a = loss_fn(model_a(X_train), y_train)
opt_a.zero_grad(); loss_a.backward(); opt_a.step()

# model_b: plain SGD + an explicit penalty in the loss
opt_b = torch.optim.SGD(model_b.parameters(), lr=0.1)
loss_b = None      # <- your loss with the penalty term
# then: zero_grad, backward, step

In [ ]:
check("weight identical", lambda: torch.allclose(model_a.weight, model_b.weight, atol=1e-7))
check("bias identical", lambda: torch.allclose(model_a.bias, model_b.bias, atol=1e-7))
torch.manual_seed(0); _m0 = nn.Linear(1, 1)
check("model_b was actually updated", lambda: not torch.allclose(model_b.weight, _m0.weight))

"Weight decay" is a literal description: each step does `w ← w − lr·(grad + λw) = (1 − lr·λ)·w − lr·grad`, i.e. every weight is first **shrunk by a constant factor** and then updated as usual.

**c)** Train a fresh `BigMLP` for 1500 epochs with `Adam(lr=0.003, weight_decay=0.01)` using your `fit`, and compare with the unregularised run.

In [ ]:
torch.manual_seed(0)
model_wd = BigMLP()
wd_train_hist, wd_val_hist = None, None

In [ ]:
check("the weights are smaller with weight decay", lambda: l2_sum(model_wd).item() < l2_sum(model_plain).item())
check("the training loss is higher (less memorisation) ...", lambda: wd_train_hist[-1] > train_hist[-1])
check("... and the validation loss at the end is lower", lambda: wd_val_hist[-1] < val_hist[-1])

Why would *small weights* generalise better? Connect it to the shape of the fitted curves you just plotted.

✍️ **Your answer** (double-click, write it in your own words *before* opening the solution):

...

---
## Exercise 9.5 · Dropout, and why `model.eval()` matters

**a)** `drop = nn.Dropout(p=0.5)` is applied to a tensor of 10 000 ones. Predict:

In [ ]:
pred_train_fraction_zero = None     # in training mode: which fraction of the outputs is 0?
pred_train_survivor_value = None    # in training mode: the value of the outputs that are not 0
pred_train_mean = None              # in training mode: the mean of the output (approximately)
pred_eval_output_value = None       # in evaluation mode: the value of every output

In [ ]:
torch.manual_seed(0)
drop = nn.Dropout(p=0.5); ones = torch.ones(10000)
drop.train(); out_train = drop(ones)
drop.eval(); out_eval = drop(ones)
check("fraction of zeros", lambda: abs(pred_train_fraction_zero - (out_train == 0).float().mean().item()) < 0.03)
check("survivor value", lambda: pred_train_survivor_value == out_train.max().item())
check("mean in training mode", lambda: abs(pred_train_mean - out_train.mean().item()) < 0.05)
check("eval mode is the identity", lambda: pred_eval_output_value == 1.0 and torch.equal(out_eval, ones))

**b)** Run the cell: the same model, the same input, called twice in each mode.

In [ ]:
torch.manual_seed(0)
m_drop = BigMLP(p_drop=0.3)
x_one = torch.tensor([[1.0]])
m_drop.train(); print("train mode:", m_drop(x_one).item(), m_drop(x_one).item())
m_drop.eval();  print("eval  mode:", m_drop(x_one).item(), m_drop(x_one).item())

What goes wrong if you (1) forget `model.eval()` before computing the validation/test metrics, (2) forget to call `model.train()` again at the start of the next epoch?

✍️ **Your answer** (double-click, write it in your own words *before* opening the solution):

...

---
## Exercise 9.6 · What the optimizers do — by hand

**a) Momentum.** PyTorch's `SGD(momentum=μ)` keeps a velocity `v` per parameter (starting at 0):

$$v \leftarrow \mu\,v + g, \qquad w \leftarrow w - \text{lr}\cdot v$$

Minimise `f(w) = w²` (gradient `g = 2w`) from `w = 1` with `lr = 0.1`, `μ = 0.9`. Compute the values of `w` after steps 1, 2 and 3 **by hand / with plain Python floats**.

In [ ]:
w_after = [None, None, None]

In [ ]:
_w = torch.tensor(1.0, requires_grad=True); _o = torch.optim.SGD([_w], lr=0.1, momentum=0.9); _ref = []
for _ in range(3):
    _o.zero_grad(); (_w ** 2).backward(); _o.step(); _ref.append(_w.item())
check(f"matches torch.optim.SGD(momentum=0.9): {np.round(_ref, 4)}", lambda: np.allclose(w_after, _ref, atol=1e-5))

Plain SGD would have gone `1 → 0.8 → 0.64 → 0.512`. With momentum the third step is much larger — the velocity has built up because all gradients pointed the same way.

**b) Adam's first step.** Adam divides the (averaged) gradient by the square root of the (averaged) squared gradient. On the very first step this ratio is `g / |g|`, so the update is

$$w \leftarrow w - \text{lr}\cdot\text{sign}(g)$$

Predict the first-step change of `w` under `Adam(lr=0.01)` for a loss whose gradient is (i) `1000`, (ii) `0.001`. Then predict the same for plain `SGD(lr=0.01)`.

In [ ]:
adam_step_big_grad = None       # change of w when the gradient is 1000
adam_step_small_grad = None     # change of w when the gradient is 0.001
sgd_step_big_grad = None
sgd_step_small_grad = None

In [ ]:
def _first_step(opt_cls, grad):
    w = torch.tensor(0.0, requires_grad=True); o = opt_cls([w], lr=0.01)
    (grad * w).backward(); o.step(); return w.item()
check("Adam, gradient 1000", lambda: abs(adam_step_big_grad - _first_step(torch.optim.Adam, 1000.0)) < 1e-6)
check("Adam, gradient 0.001", lambda: abs(adam_step_small_grad - _first_step(torch.optim.Adam, 0.001)) < 1e-6)
check("SGD, gradient 1000", lambda: abs(sgd_step_big_grad - _first_step(torch.optim.SGD, 1000.0)) < 1e-6)
check("SGD, gradient 0.001", lambda: abs(sgd_step_small_grad - _first_step(torch.optim.SGD, 0.001)) < 1e-9)

This explains two things you have already experienced:

- In notebook 03, `SGD` exploded on an unscaled feature (huge gradient → huge step). Adam's step size does not depend on the gradient's scale, so it is far more forgiving — which is why `Adam(lr=0.001)` is the default first try.
- With Adam, `lr` is roughly "how far each weight moves per step". `lr = 0.001` means it takes on the order of a thousand steps to move a weight by 1 — so reaching an output level of ~200 (unscaled prices in notebook 06) took many epochs.

**c) All three on an elongated valley.** `f(w₁, w₂) = ½(100·w₁² + w₂²)` is 100× steeper in `w₁` than in `w₂` — the picture of badly scaled features. Run the cell.

In [ ]:
def run_valley(make_opt, steps=100):
    w = torch.tensor([1.0, 1.0], requires_grad=True)
    opt = make_opt([w])
    for _ in range(steps):
        loss = 0.5 * (100 * w[0] ** 2 + w[1] ** 2)
        opt.zero_grad(); loss.backward(); opt.step()
    return (0.5 * (100 * w[0] ** 2 + w[1] ** 2)).item(), w.detach().numpy().round(4)

for name, mk in {"SGD lr=0.015": lambda p: torch.optim.SGD(p, lr=0.015),
                 "SGD lr=0.021": lambda p: torch.optim.SGD(p, lr=0.021),
                 "SGD lr=0.015 momentum=0.9": lambda p: torch.optim.SGD(p, lr=0.015, momentum=0.9),
                 "Adam lr=0.05": lambda p: torch.optim.Adam(p, lr=0.05)}.items():
    final, w = run_valley(mk)
    print(f"{name:28s} final loss {final:12.6f}   w = {w}")

Explain each of the four lines: why is plain SGD with `lr = 0.015` still far from the minimum in `w₂`? Why does `lr = 0.021` blow up (use the rule from notebook 02/03)? Why do momentum and Adam do better?

✍️ **Your answer** (double-click, write it in your own words *before* opening the solution):

...

---
## Blank-page drill (homework task I in miniature)

New empty notebook, 30 minutes. Using `load_rice()` from `dnd_utils` (4-class classification):

1. dataset + loaders, model `9 → 30 → 20 → 4`;
2. a training loop with **early stopping**: `max_epochs = 300`, `patience = 15`, monitoring the validation loss, with **automatic restoration of the best weights**;
3. a plot of training and validation loss with a vertical line at the best epoch;
4. test cross-entropy and accuracy of the restored model.

Self-check after writing: (a) is your snapshot a `copy.deepcopy`? (b) is the counter reset on improvement? (c) is `load_state_dict` called **after** the loop, so it also happens when the loop ends without triggering the patience? (d) are `model.train()` / `model.eval()` set in every epoch?

In [ ]:
# (do this one in a new notebook)